In [3]:
import os
import json
import numpy as np
import pandas as pd
import scipy.sparse as sp
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import (f1_score, classification_report,
                             roc_auc_score, multilabel_confusion_matrix)
import xgboost as xgb

In [6]:
LABELS = ["access-control", "arithmetic", "other",
          "reentrancy", "safe", "unchecked-calls"]


def load_sparse(path, name):
    z = np.load(path)
    return sp.csr_matrix((z[name + "_data"], z[name + "_indices"], z[name + "_indptr"]), shape=tuple(z[name + "_shape"]))



y_train = pd.read_parquet("/kaggle/input/datasets/mdshagorahmed2011030/pre-processed/train_preprocessed.parquet", columns=LABELS).values.astype(np.int8)
y_val = pd.read_parquet("/kaggle/input/datasets/mdshagorahmed2011030/pre-processed/validation_preprocessed.parquet", columns=LABELS).values.astype(np.int8)
y_test = pd.read_parquet("/kaggle/input/datasets/mdshagorahmed2011030/pre-processed/test_preprocessed.parquet", columns=LABELS).values.astype(np.int8)


struct_train = np.load("/kaggle/input/datasets/mdshagorahmed2011030/features-structural/train_structural_features.npz")["struct"]
struct_val = np.load("/kaggle/input/datasets/mdshagorahmed2011030/features-structural/validation_structural_features.npz")["struct"]
struct_test = np.load("/kaggle/input/datasets/mdshagorahmed2011030/features-structural/test_structural_features.npz")["struct"]


word_train = load_sparse("/kaggle/input/datasets/mdshagorahmed2011030/features-source-code/train_source_code_features.npz", "word")
word_val = load_sparse("/kaggle/input/datasets/mdshagorahmed2011030/features-source-code/validation_source_code_features.npz", "word")
word_test = load_sparse("/kaggle/input/datasets/mdshagorahmed2011030/features-source-code/test_source_code_features.npz", "word")

opcode_train = load_sparse("/kaggle/input/datasets/mdshagorahmed2011030/features-bytecode/train_bytecode_features.npz", "bcops")
opcode_val = load_sparse("/kaggle/input/datasets/mdshagorahmed2011030/features-bytecode/validation_bytecode_features.npz", "bcops")
opcode_test = load_sparse("/kaggle/input/datasets/mdshagorahmed2011030/features-bytecode/test_bytecode_features.npz", "bcops")


hand_train = np.load("/kaggle/input/datasets/mdshagorahmed2011030/features-bytecode/train_bytecode_features.npz")["bchand"]
hand_val = np.load("/kaggle/input/datasets/mdshagorahmed2011030/features-bytecode/validation_bytecode_features.npz")["bchand"]
hand_test = np.load("/kaggle/input/datasets/mdshagorahmed2011030/features-bytecode/test_bytecode_features.npz")["bchand"]


print("labels :", y_train.shape, y_val.shape, y_test.shape)
print("struct :", struct_train.shape, struct_val.shape, struct_test.shape)
print("word   :", word_train.shape, word_val.shape, word_test.shape)
print("opcode :", opcode_train.shape, opcode_val.shape, opcode_test.shape)
print("hand   :", hand_train.shape, hand_val.shape, hand_test.shape)


labels : (79641, 6) (10861, 6) (15972, 6)
struct : (79641, 52) (10861, 52) (15972, 52)
word   : (79641, 20000) (10861, 20000) (15972, 20000)
opcode : (79641, 12000) (10861, 12000) (15972, 12000)
hand   : (79641, 62) (10861, 62) (15972, 62)


In [25]:
print("XGBoost on byte_code features")
xgb_word = xgb.XGBClassifier(
    objective="binary:logistic",
    n_estimators=600,
    max_depth=8,
    learning_rate=0.086,
    subsample=0.78,
    colsample_bytree=0.40,
    min_child_weight=10,
    reg_lambda=0.37,
    reg_alpha=2.83,
    gamma=0.99,
    tree_method="hist",
    max_bin=64,                      
    eval_metric="logloss",
    early_stopping_rounds=60,
    random_state=RANDOM_STATE,
    n_jobs=4,
    verbosity=0,
)

xgb_word.fit(
    word_train, y_train,
    eval_set=[(word_val, y_val)],
    verbose=25,              
)

print("   stopped at round", xgb_word.best_iteration)
report("xgb_word",
       get_probs(xgb_word, word_val),
       get_probs(xgb_word, word_test))
xgb_word.save_model(os.path.join(OUT_MODELS, "xgb_word.json"))
print("   model saved.")

[0]	validation_0-logloss:0.51575
[25]	validation_0-logloss:0.19470
[50]	validation_0-logloss:0.15272
[75]	validation_0-logloss:0.13879
[100]	validation_0-logloss:0.13071
[125]	validation_0-logloss:0.12509
[150]	validation_0-logloss:0.12027
[175]	validation_0-logloss:0.11639
[200]	validation_0-logloss:0.11320
[225]	validation_0-logloss:0.11023
[250]	validation_0-logloss:0.10775
[275]	validation_0-logloss:0.10536
[300]	validation_0-logloss:0.10351
[325]	validation_0-logloss:0.10180
[350]	validation_0-logloss:0.10029
[375]	validation_0-logloss:0.09891
[400]	validation_0-logloss:0.09778
[425]	validation_0-logloss:0.09672
[450]	validation_0-logloss:0.09577
[475]	validation_0-logloss:0.09495
[500]	validation_0-logloss:0.09424
[525]	validation_0-logloss:0.09361
[550]	validation_0-logloss:0.09312
[575]	validation_0-logloss:0.09275
[599]	validation_0-logloss:0.09244
   stopped at round 599

   val  macro-F1 : 0.9301
   test macro-F1 : 0.9318
   test micro-F1 : 0.9397
   test AUC      : 0.9923



In [26]:
print("XGBoost on structural features")
xgb_struct = xgb.XGBClassifier(
    objective="binary:logistic",
    n_estimators=600,
    max_depth=10,
    learning_rate=0.031,
    subsample=0.68,
    colsample_bytree=0.88,
    min_child_weight=5,
    reg_lambda=0.23,
    reg_alpha=0.68,
    gamma=0.51,
    tree_method="hist",
    eval_metric="logloss",
    early_stopping_rounds=60,
    random_state=RANDOM_STATE,
    n_jobs=4,
    verbosity=0,
)

xgb_struct.fit(
    struct_train, y_train,
    eval_set=[(struct_val, y_val)],
    verbose=25,              # validation logloss every 25 rounds
)

report("xgb_struct",
       get_probs(xgb_struct, struct_val),
       get_probs(xgb_struct, struct_test))
xgb_struct.save_model(os.path.join(OUT_MODELS, "xgb_struct.json"))
print("   model saved.")

XGBoost on structural features
[0]	validation_0-logloss:0.54691
[25]	validation_0-logloss:0.31679
[50]	validation_0-logloss:0.23511
[75]	validation_0-logloss:0.19770
[100]	validation_0-logloss:0.17850
[125]	validation_0-logloss:0.16754
[150]	validation_0-logloss:0.16116
[175]	validation_0-logloss:0.15708
[200]	validation_0-logloss:0.15400
[225]	validation_0-logloss:0.15133
[250]	validation_0-logloss:0.14915
[275]	validation_0-logloss:0.14696
[300]	validation_0-logloss:0.14502
[325]	validation_0-logloss:0.14329
[350]	validation_0-logloss:0.14154
[375]	validation_0-logloss:0.14002
[400]	validation_0-logloss:0.13863
[425]	validation_0-logloss:0.13730
[450]	validation_0-logloss:0.13605
[475]	validation_0-logloss:0.13496
[500]	validation_0-logloss:0.13384
[525]	validation_0-logloss:0.13290
[550]	validation_0-logloss:0.13197
[575]	validation_0-logloss:0.13104
[599]	validation_0-logloss:0.13024

   val  macro-F1 : 0.8984
   test macro-F1 : 0.8990
   test micro-F1 : 0.9116
   test AUC      : 0

In [27]:
print("XGBoost on opcode tf-idf")
xgb_opcode = xgb.XGBClassifier(
    objective="binary:logistic",
    n_estimators=600,
    max_depth=8,
    learning_rate=0.050,
    subsample=0.93,
    colsample_bytree=0.72,
    min_child_weight=2,
    reg_lambda=0.61,
    reg_alpha=1.85,
    gamma=2.23,
    tree_method="hist",
    max_bin=64,
    eval_metric="logloss",
    early_stopping_rounds=60,
    random_state=RANDOM_STATE,
    n_jobs=4,
    verbosity=0,
)

xgb_opcode.fit(
    opcode_train, y_train,
    eval_set=[(opcode_val, y_val)],
    verbose=25,             
)
report("xgb_opcode",
       get_probs(xgb_opcode, opcode_val),
       get_probs(xgb_opcode, opcode_test))
xgb_opcode.save_model(os.path.join(OUT_MODELS, "xgb_opcode.json"))
print("   model saved.")

XGBoost on opcode tf-idf
[0]	validation_0-logloss:0.53657
[25]	validation_0-logloss:0.26075
[50]	validation_0-logloss:0.19741
[75]	validation_0-logloss:0.17484
[100]	validation_0-logloss:0.16443
[125]	validation_0-logloss:0.15788
[150]	validation_0-logloss:0.15304
[175]	validation_0-logloss:0.14905
[200]	validation_0-logloss:0.14554
[225]	validation_0-logloss:0.14270
[250]	validation_0-logloss:0.14009
[275]	validation_0-logloss:0.13800
[300]	validation_0-logloss:0.13619
[325]	validation_0-logloss:0.13435
[350]	validation_0-logloss:0.13280
[375]	validation_0-logloss:0.13153
[400]	validation_0-logloss:0.13033
[425]	validation_0-logloss:0.12937
[450]	validation_0-logloss:0.12845
[475]	validation_0-logloss:0.12768
[500]	validation_0-logloss:0.12707
[525]	validation_0-logloss:0.12664
[550]	validation_0-logloss:0.12637
[575]	validation_0-logloss:0.12615
[599]	validation_0-logloss:0.12594

   val  macro-F1 : 0.9011
   test macro-F1 : 0.9032
   test micro-F1 : 0.9160
   test AUC      : 0.9855


In [28]:

print("XGBoost on hand-written opcode features")
xgb_hand = xgb.XGBClassifier(
    objective="binary:logistic",
    n_estimators=600,
    max_depth=9,
    learning_rate=0.067,
    subsample=0.70,
    colsample_bytree=0.74,
    min_child_weight=3,
    reg_lambda=6.76,
    reg_alpha=1.95,
    gamma=1.15,
    tree_method="hist",
    eval_metric="logloss",
    early_stopping_rounds=60,
    random_state=RANDOM_STATE,
    n_jobs=4,
    verbosity=0,
)

xgb_hand.fit(
    hand_train, y_train,
    eval_set=[(hand_val, y_val)],
    verbose=25,              
)

report("xgb_hand",
       get_probs(xgb_hand, hand_val),
       get_probs(xgb_hand, hand_test))
xgb_hand.save_model(os.path.join(OUT_MODELS, "xgb_hand.json"))
print("   model saved.")

XGBoost on hand-written opcode features
[0]	validation_0-logloss:0.53322
[25]	validation_0-logloss:0.26908
[50]	validation_0-logloss:0.22287
[75]	validation_0-logloss:0.20651
[100]	validation_0-logloss:0.19709
[125]	validation_0-logloss:0.19053
[150]	validation_0-logloss:0.18561
[175]	validation_0-logloss:0.18153
[200]	validation_0-logloss:0.17850
[225]	validation_0-logloss:0.17576
[250]	validation_0-logloss:0.17351
[275]	validation_0-logloss:0.17156
[300]	validation_0-logloss:0.16986
[325]	validation_0-logloss:0.16853
[350]	validation_0-logloss:0.16752
[375]	validation_0-logloss:0.16659
[400]	validation_0-logloss:0.16587
[425]	validation_0-logloss:0.16525
[450]	validation_0-logloss:0.16478
[475]	validation_0-logloss:0.16429
[500]	validation_0-logloss:0.16402
[525]	validation_0-logloss:0.16370
[550]	validation_0-logloss:0.16354
[575]	validation_0-logloss:0.16332
[599]	validation_0-logloss:0.16313

   val  macro-F1 : 0.8631
   test macro-F1 : 0.8674
   test micro-F1 : 0.8835
   test AUC

In [8]:
print(" MLP on source text")
mlp_word = MLPClassifier(
    hidden_layer_sizes=(512, 256),
    activation="relu",
    alpha=1e-4,
    batch_size=256,
    learning_rate_init=1e-3,
    max_iter=40,
    early_stopping=True,
    n_iter_no_change=6,
    random_state=RANDOM_STATE,
    verbose=True,
)

mlp_word.fit(word_train, y_train)
report("mlp_word",
       get_probs(mlp_word, word_val),
       get_probs(mlp_word, word_test))
joblib.dump(mlp_word, os.path.join(OUT_MODELS, "mlp_word.joblib"))
print("   model saved.")

 MLP on source text
Iteration 1, loss = 1.43409359
Validation score: 0.715756
Iteration 2, loss = 0.90678392
Validation score: 0.755053
Iteration 3, loss = 0.71755027
Validation score: 0.775392
Iteration 4, loss = 0.59189427
Validation score: 0.793597
Iteration 5, loss = 0.49242203
Validation score: 0.809542
Iteration 6, loss = 0.41874718
Validation score: 0.808161
Iteration 7, loss = 0.35751285
Validation score: 0.822599
Iteration 8, loss = 0.31360230
Validation score: 0.823854
Iteration 9, loss = 0.28168180
Validation score: 0.829002
Iteration 10, loss = 0.25188644
Validation score: 0.830383
Iteration 11, loss = 0.22977679
Validation score: 0.830760
Iteration 12, loss = 0.21234528
Validation score: 0.835028
Iteration 13, loss = 0.19612258
Validation score: 0.835530
Iteration 14, loss = 0.18791755
Validation score: 0.843189
Iteration 15, loss = 0.17956436
Validation score: 0.836033
Iteration 16, loss = 0.16057126
Validation score: 0.835530
Iteration 17, loss = 0.15937764
Validation sc

In [9]:
print("MLP on structural features")
mlp_struct = MLPClassifier(
    hidden_layer_sizes=(512, 256),
    activation="relu",
    alpha=1e-4,
    batch_size=256,
    learning_rate_init=1e-3,
    max_iter=40,
    early_stopping=True,
    n_iter_no_change=6,
    random_state=RANDOM_STATE,
    verbose=True,
)

mlp_struct.fit(struct_train, y_train)

report("mlp_struct",
       get_probs(mlp_struct, struct_val),
       get_probs(mlp_struct, struct_test))
joblib.dump(mlp_struct, os.path.join(OUT_MODELS, "mlp_struct.joblib"))
print("   model saved.")

MLP on structural features
Iteration 1, loss = 1.89976220
Validation score: 0.583427
Iteration 2, loss = 1.53021618
Validation score: 0.617577
Iteration 3, loss = 1.39326703
Validation score: 0.639799
Iteration 4, loss = 1.30228356
Validation score: 0.643942
Iteration 5, loss = 1.23480621
Validation score: 0.667169
Iteration 6, loss = 1.17804619
Validation score: 0.684871
Iteration 7, loss = 1.13103407
Validation score: 0.693032
Iteration 8, loss = 1.08868826
Validation score: 0.685374
Iteration 9, loss = 1.05231917
Validation score: 0.696171
Iteration 10, loss = 1.01698633
Validation score: 0.706215
Iteration 11, loss = 0.98587364
Validation score: 0.709479
Iteration 12, loss = 0.96439912
Validation score: 0.712367
Iteration 13, loss = 0.93203328
Validation score: 0.720276
Iteration 14, loss = 0.90732140
Validation score: 0.722913
Iteration 15, loss = 0.88643888
Validation score: 0.735719
Iteration 16, loss = 0.86257931
Validation score: 0.730948
Iteration 17, loss = 0.84072203
Valida

/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (40) reached and the optimization hasn't converged yet.
  warnings.warn(


                 precision    recall  f1-score   support

 access-control       0.85      0.84      0.84      2346
     arithmetic       0.87      0.81      0.84      2715
          other       0.87      0.83      0.85      4209
     reentrancy       0.92      0.86      0.89      4846
           safe       0.91      0.90      0.91      5422
unchecked-calls       0.94      0.93      0.93      7289

      micro avg       0.90      0.88      0.89     26827
      macro avg       0.89      0.86      0.88     26827
   weighted avg       0.90      0.88      0.89     26827
    samples avg       0.88      0.87      0.87     26827

   saved mlp_struct_val.npy and mlp_struct_test.npy
   model saved.


In [10]:
print("MLP on hand-written opcode features")
mlp_hand = MLPClassifier(
    hidden_layer_sizes=(512, 256),
    activation="relu",
    alpha=1e-4,
    batch_size=256,
    learning_rate_init=1e-3,
    max_iter=40,
    early_stopping=True,
    n_iter_no_change=6,
    random_state=RANDOM_STATE,
    verbose=True,
)

mlp_hand.fit(hand_train, y_train)

report("mlp_hand",
       get_probs(mlp_hand, hand_val),
       get_probs(mlp_hand, hand_test))
joblib.dump(mlp_hand, os.path.join(OUT_MODELS, "mlp_hand.joblib"))
print("   model saved.")

MLP on hand-written opcode features
Iteration 1, loss = 1.97913185
Validation score: 0.553045
Iteration 2, loss = 1.64654948
Validation score: 0.593095
Iteration 3, loss = 1.52726243
Validation score: 0.621469
Iteration 4, loss = 1.44735264
Validation score: 0.645198
Iteration 5, loss = 1.37741511
Validation score: 0.657878
Iteration 6, loss = 1.32674659
Validation score: 0.659259
Iteration 7, loss = 1.27612123
Validation score: 0.668550
Iteration 8, loss = 1.23083252
Validation score: 0.674827
Iteration 9, loss = 1.19549408
Validation score: 0.682109
Iteration 10, loss = 1.15995908
Validation score: 0.695041
Iteration 11, loss = 1.12431650
Validation score: 0.693409
Iteration 12, loss = 1.09417376
Validation score: 0.689014
Iteration 13, loss = 1.06593745
Validation score: 0.697301
Iteration 14, loss = 1.03837836
Validation score: 0.687006
Iteration 15, loss = 1.01456722
Validation score: 0.703955
Iteration 16, loss = 0.98621438
Validation score: 0.711237
Iteration 17, loss = 0.965751

In [ ]:
print("MLP on opcode tf-idf")
mlp_opcode = MLPClassifier(
    hidden_layer_sizes=(512, 256),
    activation="relu",
    alpha=1e-4,
    batch_size=256,
    learning_rate_init=1e-3,
    max_iter=40,
    early_stopping=True,
    n_iter_no_change=6,
    random_state=RANDOM_STATE,
    verbose=True,
)

mlp_opcode.fit(opcode_train, y_train)

print("   stopped after", mlp_opcode.n_iter_, "epochs")
report("mlp_opcode",
       get_probs(mlp_opcode, opcode_val),
       get_probs(mlp_opcode, opcode_test))
joblib.dump(mlp_opcode, os.path.join(OUT_MODELS, "mlp_opcode.joblib"))
print("   model saved.")

In [11]:
print("Logistic regression on source text")
logreg_word = OneVsRestClassifier(
    LogisticRegression(
        C=1.0,
        solver="liblinear",
        class_weight="balanced",     
        max_iter=1000,
        random_state=RANDOM_STATE,
    ),
    n_jobs=1,
)

logreg_word.fit(word_train, y_train)

report("logreg_word",
       get_probs(logreg_word, word_val),
       get_probs(logreg_word, word_test))
joblib.dump(logreg_word, os.path.join(OUT_MODELS,"logreg_word.joblib"))
print("   model saved.")


Logistic regression on source text

   val  macro-F1 : 0.8441
   test macro-F1 : 0.8470
   test micro-F1 : 0.8651
   test AUC      : 0.9743

                 precision    recall  f1-score   support

 access-control       0.68      0.88      0.77      2346
     arithmetic       0.72      0.90      0.80      2715
          other       0.75      0.89      0.82      4209
     reentrancy       0.85      0.89      0.87      4846
           safe       0.86      0.93      0.90      5422
unchecked-calls       0.94      0.92      0.93      7289

      micro avg       0.83      0.91      0.87     26827
      macro avg       0.80      0.90      0.85     26827
   weighted avg       0.84      0.91      0.87     26827
    samples avg       0.85      0.90      0.86     26827

   saved logreg_word_val.npy and logreg_word_test.npy
   model saved.


In [12]:
print("Logistic regression on opcode tf-idf")
logreg_opcode = OneVsRestClassifier(
    LogisticRegression(
        C=1.0,
        solver="liblinear",
        class_weight="balanced",
        max_iter=1000,
        random_state=RANDOM_STATE,
    ),
    n_jobs=1,
)
logreg_opcode.fit(opcode_train, y_train)
report("logreg_opcode",
       get_probs(logreg_opcode, opcode_val),
       get_probs(logreg_opcode, opcode_test))
joblib.dump(logreg_opcode, os.path.join(OUT_MODELS,"logreg_opcode.joblib"))
print("   model saved.")

Logistic regression on opcode tf-idf

   val  macro-F1 : 0.8137
   test macro-F1 : 0.8133
   test micro-F1 : 0.8355
   test AUC      : 0.9620

                 precision    recall  f1-score   support

 access-control       0.62      0.87      0.72      2346
     arithmetic       0.63      0.88      0.74      2715
          other       0.70      0.85      0.77      4209
     reentrancy       0.83      0.88      0.85      4846
           safe       0.85      0.92      0.88      5422
unchecked-calls       0.91      0.92      0.92      7289

      micro avg       0.78      0.89      0.84     26827
      macro avg       0.76      0.89      0.81     26827
   weighted avg       0.80      0.89      0.84     26827
    samples avg       0.82      0.89      0.84     26827

   saved logreg_opcode_val.npy and logreg_opcode_test.npy
   model saved.


In [7]:

MODEL_NAMES = ["xgb_word", "xgb_struct", "xgb_opcode", "xgb_hand",
               "mlp_word", "mlp_struct", "mlp_opcode", "mlp_hand",
               "logreg_word", "logreg_opcode"]

rows = []
for name in MODEL_NAMES:
    path = os.path.join(OUT_PROBS, name + "_test.npy")
    if not os.path.exists(path):
        continue
    probs = np.load(path)
    pred = (probs >= 0.5).astype(int)
    rows.append({
        "model": name,
        "test_macro_f1": round(f1_score(y_test, pred, average="macro",
                                        zero_division=0), 4),
        "test_micro_f1": round(f1_score(y_test, pred, average="micro",
                                        zero_division=0), 4),
        "test_auc": round(roc_auc_score(y_test, probs,
                                        average="macro"), 4),
    })

summary = pd.DataFrame(rows).sort_values("test_macro_f1",
                                         ascending=False)
summary.index = range(1, len(summary) + 1)
print(summary.to_string())

summary.to_csv(os.path.join(WORK, "model_summary.csv"), index=False)



            model  test_macro_f1  test_micro_f1  test_auc
1        xgb_word         0.9318         0.9397    0.9923
2        mlp_word         0.9198         0.9288    0.9839
3      xgb_opcode         0.9032         0.9160    0.9855
4      xgb_struct         0.8990         0.9116    0.9849
5      mlp_opcode         0.8924         0.9057    0.9720
6      mlp_struct         0.8769         0.8902    0.9705
7        xgb_hand         0.8674         0.8835    0.9750
8     logreg_word         0.8470         0.8651    0.9743
9        mlp_hand         0.8372         0.8575    0.9594
10  logreg_opcode         0.8133         0.8355    0.9620


In [24]:
def get_probs(model, X):
    probs = model.predict_proba(X)
    if isinstance(probs, list):           # OneVsRestClassifier style
        probs = np.column_stack([q[:, 1] for q in probs])
    return np.asarray(probs).reshape(X.shape[0], -1)


def report(name, val_probs, test_probs, save_probs=True):
    val_pred = (val_probs >= 0.5).astype(int)
    test_pred = (test_probs >= 0.5).astype(int)

    print()
    print("   val  macro-F1 : %.4f" % f1_score(y_val, val_pred,
                                               average="macro",
                                               zero_division=0))
    print("   test macro-F1 : %.4f" % f1_score(y_test, test_pred,
                                               average="macro",
                                               zero_division=0))
    print("   test micro-F1 : %.4f" % f1_score(y_test, test_pred,
                                               average="micro",
                                               zero_division=0))
    print("   test AUC      : %.4f" % roc_auc_score(y_test, test_probs,
                                                    average="macro"))
    print()
    print(classification_report(y_test, test_pred,
                                target_names=CLASS_NAMES,
                                zero_division=0))
    if save_probs:
        np.save(os.path.join(OUT_PROBS, name + "_val.npy"),
                val_probs.astype(np.float32))
        np.save(os.path.join(OUT_PROBS, name + "_test.npy"),
                test_probs.astype(np.float32))
        print("   saved", name + "_val.npy", "and",
              name + "_test.npy")

In [ ]:
RANDOM_STATE = 42
D_PRE = "/kaggle/input/datasets/mdshagorahmed2011030/pre-processed"
D_STRUCT = "/kaggle/input/datasets/mdshagorahmed2011030/features-structural"
D_WORD = "/kaggle/input/datasets/mdshagorahmed2011030/features-source-code"
D_BYTE = "/kaggle/input/datasets/mdshagorahmed2011030/features-bytecode"
WORK = "/kaggle/working"
OUT_MODELS = os.path.join(WORK, "models")
OUT_PROBS = os.path.join(WORK, "probs")
os.makedirs(OUT_MODELS, exist_ok=True)
os.makedirs(OUT_PROBS, exist_ok=True)
np.random.seed(RANDOM_STATE)

In [ ]:
train_df = pd.read_parquet(D_PRE + "/train_preprocessed.parquet")
val_df = pd.read_parquet(D_PRE + "/validation_preprocessed.parquet")
test_df = pd.read_parquet(D_PRE + "/test_preprocessed.parquet")

NOT_A_LABEL = ["address", "source_capped", "opcodes", "src_hash",
               "bc_hash", "src_len", "bc_len", "trunc_flag",
               "bc_trunc_flag"]
CLASS_NAMES = [c for c in train_df.columns
               if c not in NOT_A_LABEL
               and train_df[c].dtype.kind in "biu"]      
N_CLASSES = len(CLASS_NAMES)

skipped = [c for c in train_df.columns if c not in CLASS_NAMES]
assert N_CLASSES >= 2, "no 0/1 label columns found in the parquet"

y_train = train_df[CLASS_NAMES].values.astype(np.int8)
y_val = val_df[CLASS_NAMES].values.astype(np.int8)
y_test = test_df[CLASS_NAMES].values.astype(np.int8)

# ---- structural features: plain dense arrays ----
struct_train = np.load(D_STRUCT + "/train_structural_features.npz")["struct"]
struct_val = np.load(D_STRUCT + "/validation_structural_features.npz")["struct"]
struct_test = np.load(D_STRUCT + "/test_structural_features.npz")["struct"]


def load_sparse(path, name):
    z = np.load(path)
    return sp.csr_matrix((z[name + "_data"], z[name + "_indices"],
                          z[name + "_indptr"]),
                         shape=tuple(z[name + "_shape"]))



word_train = load_sparse(D_WORD + "/train_source_code_features.npz", "word")
word_val = load_sparse(D_WORD + "/validation_source_code_features.npz", "word")
word_test = load_sparse(D_WORD + "/test_source_code_features.npz", "word")


#      tf-idf and the dense hand-written columns ----
opcode_train = load_sparse(D_BYTE + "/train_bytecode_features.npz", "bcops")
opcode_val = load_sparse(D_BYTE + "/validation_bytecode_features.npz", "bcops")
opcode_test = load_sparse(D_BYTE + "/test_bytecode_features.npz", "bcops")

hand_train = np.load(D_BYTE + "/train_bytecode_features.npz")["bchand"]
hand_val = np.load(D_BYTE + "/validation_bytecode_features.npz")["bchand"]
hand_test = np.load(D_BYTE + "/test_bytecode_features.npz")["bchand"]

print("   classes :", N_CLASSES, CLASS_NAMES)
print("   rows    : train %d, val %d, test %d" % (
    len(y_train), len(y_val), len(y_test)))
print("   word    :", word_train.shape)
print("   struct  :", struct_train.shape)
print("   opcode  :", opcode_train.shape)
print("   hand    :", hand_train.shape)
